In [1]:
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
import random
import math
import time
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

# If you use rich for better logs:
try:
    from rich import print as rprint
    from rich.table import Table
except Exception:
    rprint = print

# PyTorch Geometric (hetero)
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as PYGDataLoader
from torch_geometric.nn import GINConv, global_mean_pool

In [2]:
BASE_DIR = Path(os.environ.get("HEROSIM_BASE_DIR", "/root/projects/my-herosim"))
DATASETS_DIR = BASE_DIR / "simulation_data/gnn_datasets"
RESULTS_DIR  = BASE_DIR / "simulation_data/initial_results_simple"

print(f"BASE_DIR    = {BASE_DIR}")
print(f"DATASETS_DIR= {DATASETS_DIR}")
print(f"RESULTS_DIR = {RESULTS_DIR}")

BASE_DIR    = /root/projects/my-herosim
DATASETS_DIR= /root/projects/my-herosim/simulation_data/gnn_datasets
RESULTS_DIR = /root/projects/my-herosim/simulation_data/initial_results_simple


In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # no-op if no CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [4]:
# %%
# Data assumptions & extraction strategy
# -------------------------------------
# We will build supervision from "result" JSONs emitted by your simulator runs.
# For each result:
#   - Detect all tasks in the workload sample.
#   - Detect all candidate/feasible platforms for each task (edges).
#   - Detect the "chosen" (ground-truth) platform per task (label).
#   - Optionally extract simple numeric features for tasks/platforms (or use ID embeddings).
#
# The exact JSON schema can vary; we make liberal, defensive attempts to find:
#   result['sample']['placement_plan']                 -> ground truth (mapping task_id -> platform_id)
#   result['stats']['taskResults'][i]['placement']     -> backup (per-task chosen placement)
#   result['sample']['feasible'] / ['feasible_map']    -> candidate neighbors
#
# If feasible edges are not explicitly stored, we try to reconstruct from:
#   - sample['feasible_platforms_by_task'] (if present)
#   - else: fall back to unique platforms mentioned in placement_plan (but that loses negatives)
#
# IMPORTANT: Superb results come from having the full feasible sets. If missing, consider
# emitting them into your result JSONs at generation time.

In [5]:
# %%
# Small helpers to find likely fields in JSON safely

def _get(d: dict, path: List[str], default=None):
    cur = d
    for k in path:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def read_json(path: Path) -> Optional[dict]:
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] Could not read JSON {path}: {e}")
        return None

In [16]:
# %%
# Find the smallest optimal_result.json across all ds_xxxx folders
# ---------------------------------------------------------------

from pathlib import Path

DATASETS_DIR = Path("/root/projects/my-herosim/simulation_data/gnn_datasets")

smallest_file = None
smallest_size = float("inf")

for ds_dir in sorted(DATASETS_DIR.glob("ds_*")):
    opt_file = ds_dir / "optimal_result.json"
    if opt_file.is_file():
        size = opt_file.stat().st_size
        if size < smallest_size:
            smallest_size = size
            smallest_file = opt_file

if smallest_file:
    print(f"Smallest optimal_result.json: {smallest_file} ({smallest_size} bytes)")
else:
    print("No optimal_result.json files found.")

Smallest optimal_result.json: /root/projects/my-herosim/simulation_data/gnn_datasets/ds_0049/optimal_result.json (125696 bytes)


In [19]:
# %%
# Inspect structure of an optimal_result.json and summarize key info (robust to list placements)
# ----------------------------------------------------------------------------------------------

import json
from pathlib import Path

OPT_PATH = Path("/root/projects/my-herosim/simulation_data/gnn_datasets/ds_0049/optimal_result.json")

with open(OPT_PATH, "r") as f:
    result = json.load(f)

print(f"Loaded: {OPT_PATH} ({OPT_PATH.stat().st_size} bytes)\n")

def safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

print("Top-level keys:", list(result.keys()), "\n")

sample = result.get("sample", {})
stats  = result.get("stats", {})

placement_plan = safe_get(result, "sample", "placement_plan", default={})
network         = safe_get(result, "sample", "network", default={})
topology        = safe_get(result, "sample", "network", "topology", default={})
rtt_total       = safe_get(result, "stats", "metrics", "rtt_total", default=None) or \
                  safe_get(result, "stats", "rtt", default=None) or \
                  safe_get(result, "stats", "Total RTT", default=None)

# --- Tasks & Platforms ---
tasks = list(placement_plan.keys())

# Flatten possible list values
platforms_flat = []
for v in placement_plan.values():
    if isinstance(v, list):
        platforms_flat.extend(v)
    else:
        platforms_flat.append(v)
platforms = list({str(p) for p in platforms_flat if p is not None})

print(f"Tasks: {len(tasks)}\nPlatforms: {len(platforms)}")

if placement_plan:
    print("\nExample placements (task → platform[s]):")
    for t, p in list(placement_plan.items())[:10]:
        print(f"  {t:25s} → {p}")
    if len(placement_plan) > 10:
        print(f"  ... ({len(placement_plan)} total mappings)")

# --- Network topology ---
if topology:
    nodes = topology.get("nodes", {})
    edges = topology.get("edges", {}) or topology.get("links", {})
    print(f"\nNetwork topology:")
    print(f"  Nodes: {len(nodes)}")
    if isinstance(edges, dict):
        print(f"  Edges: {sum(len(v) for v in edges.values())}")
    elif isinstance(edges, list):
        print(f"  Edges: {len(edges)}")

    # show a few sample edges with latency if present
    shown = 0
    print("  Example edges:")
    if isinstance(edges, dict):
        for src, dsts in edges.items():
            if not isinstance(dsts, dict):
                continue
            for dst, meta in dsts.items():
                lat = (meta.get("latency") or meta.get("delay") or "?") if isinstance(meta, dict) else "?"
                print(f"    {src} → {dst}  latency={lat}")
                shown += 1
                if shown >= 5:
                    break
            if shown >= 5:
                break
    elif isinstance(edges, list):
        for e in edges[:5]:
            print("   ", e)

# --- Metrics ---
print("\n--- Metrics ---")
if isinstance(stats, dict):
    for k, v in stats.items():
        if any(word in k.lower() for word in ["rtt", "latency", "energy", "penalty"]):
            print(f"{k:25s}: {v}")
else:
    print(stats)

if rtt_total:
    print(f"\nTotal RTT: {rtt_total}")
else:
    print("\nTotal RTT: not found")

print("\nDone.")

Loaded: /root/projects/my-herosim/simulation_data/gnn_datasets/ds_0049/optimal_result.json (125696 bytes)

Top-level keys: ['status', 'config', 'sim_inputs', 'stats', 'sample'] 

Tasks: 5
Platforms: 8

Example placements (task → platform[s]):
  0                         → [11, 53]
  1                         → [14, 68]
  2                         → [14, 69]
  3                         → [11, 54]
  4                         → [1, 4]

--- Metrics ---
averageColdStartTime     : 0.057600000000000005
penaltyProportion        : 0.0
penaltyDistributionOverTime: []
energy                   : 7.027258579947099e-06
reclaimableEnergy        : 4.8429135163801985e-06
averageNetworkLatency    : 0.06044067513238014

Total RTT: not found

Done.


In [ ]:
# %%
# FIX v2: dataset discovery — only use ds_XXXX whose results_index.txt is non-empty
# and DIFFERENT from ds_{XXXX-1}. Then take the LAST non-empty, non-PDF line and
# resolve it strictly under RESULTS_DIR (initial_results_simple).
# -------------------------------------------------------------------------------

from pathlib import Path
from typing import List

def collect_result_files_last_only_changed_indices(datasets_dir: Path, results_dir: Path) -> List[Path]:
    result_paths: List[Path] = []
    # sort lexicographically; ds_0001, ds_0002, ...
    ds_dirs = sorted([p for p in datasets_dir.glob("ds_*") if p.is_dir()])

    prev_index_content = None  # used to compare with immediate predecessor

    i = 1
    for ds in ds_dirs:
        idx_file = ds / "results_index.txt"
        if not idx_file.exists():
            continue

        try:
            # raw content (keep exact text for equality check)
            raw = idx_file.read_text()
        except Exception:
            continue

        # Skip empty files
        if not raw.strip():
            continue

        # Only accept if content differs from the previous ds' index
        if prev_index_content is not None and raw == prev_index_content:
            # same as previous → skip
            prev_index_content = raw  # still update to keep chain correct
            continue

        # Store for next comparison
        prev_index_content = raw

        # Parse lines and pick the last relevant JSON entry
        lines = [ln.strip() for ln in raw.splitlines()]
        candidates = [ln for ln in lines if ln and not ln.lower().endswith(".pdf")]
        if not candidates:
            continue

        last_line = candidates[-1]

        # Always resolve under RESULTS_DIR (initial_results_simple)
        # If last_line is absolute, convert it to just its name relative to results dir.
        p = Path(last_line)
        if p.is_absolute():
            p = Path(p.name)  # keep just the filename if absolute was mistakenly logged
        p = results_dir / p

        if p.suffix.lower() == ".json" and p.exists():
            result_paths.append(p)

    # Deduplicate while preserving order
    seen = set()
    unique_paths = []
    for p in result_paths:
        if p not in seen:
            seen.add(p)
            unique_paths.append(p)

    return unique_paths

# Build the list (optionally cap via MAX_RESULT_FILES env)
all_result_files = collect_result_files_last_only_changed_indices(DATASETS_DIR, RESULTS_DIR)
MAX_FILES = int(os.environ.get("MAX_RESULT_FILES", "0"))  # 0 -> no cap
picked_files = all_result_files if MAX_FILES <= 0 else all_result_files[:MAX_FILES]

print(f"Datasets discovered: {len(picked_files)} (changed indices; last line from each)")
if picked_files[:5]:
    print("Sample:", *[pf.name for pf in picked_files[:5]], sep="\n  - ")

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [8]:
# %%
# Graph building
# --------------
# We'll create a hetero graph with node types:
#   - 'task': N_t nodes
#   - 'platform': N_p nodes
# Edges:
#   - ('task','feasible','platform'): one edge per feasible (t, p)
#
# Node features:
#   - If you have real features, inject them in `build_node_features`.
#   - Otherwise we provide minimal learnable embeddings by ID + type encoders.
#
# Labels:
#   - For each task t, we want the index of the correct platform among N(t) neighbors.
#   - We'll store:
#       * y_task_platform[t] = platform_id (global platform index)
#       * edge masks derived from the bipartite incidence to enable masked softmax.


@dataclass
class GraphSample:
    data: HeteroData
    task_id_map: Dict[Any, int]         # original task id -> 0..T-1
    platform_id_map: Dict[Any, int]     # original platform id -> 0..P-1
    task_gt_platform: Dict[int, int]    # local task index -> local platform index (ground-truth)
    meta: Dict[str, Any]                # optional metadata (file path, dataset id, etc.)


def discover_ids_and_edges(result_json: dict) -> Tuple[List[Any], List[Any], List[Tuple[Any, Any]], Dict[Any, Any]]:
    """
    Return (task_ids, platform_ids, feasible_edges, gt_map)
      - task_ids:    list of raw task IDs
      - platform_ids:list of raw platform IDs
      - feasible_edges: list of (task_id, platform_id)
      - gt_map: mapping {task_id: chosen_platform_id} as ground-truth
    """
    sample = result_json.get("sample", {}) or {}

    # 1) Ground-truth assignments (prefer explicit placement_plan)
    gt_map = _get(result_json, ["sample", "placement_plan"], default=None)
    if gt_map is None:
        # Try to reconstruct from stats.taskResults
        gt_map = {}
        task_results = _get(result_json, ["stats", "taskResults"], default=[]) or []
        for tr in task_results:
            tid = tr.get("id") or tr.get("taskId") or tr.get("name") or tr.get("task_id")
            # placement can be nested; try typical patterns
            placement = tr.get("placement") or tr.get("platform") or tr.get("server") or {}
            pid = (
                placement if isinstance(placement, str) else
                placement.get("id") if isinstance(placement, dict) else
                placement.get("name") if isinstance(placement, dict) else None
            )
            if tid is not None and pid is not None:
                gt_map[tid] = pid

    if not gt_map or len(gt_map) == 0:
        raise ValueError("Could not find ground-truth placement for tasks in this result JSON.")

    # Raw task IDs
    task_ids = list(gt_map.keys())

    # 2) Feasible edges (prefer explicit feasible data)
    feasible_edges: List[Tuple[Any, Any]] = []

    # Common places a feasible map might live:
    # - sample['feasible'] : {task_id: [platform_ids]}
    # - sample['feasible_platforms_by_task'] : same idea
    # - sample['feasible_map']
    feas_map = (
        sample.get("feasible")
        or sample.get("feasible_platforms_by_task")
        or sample.get("feasible_map")
        or None
    )

    # Collect platforms seen
    platform_ids_set = set()

    if isinstance(feas_map, dict) and len(feas_map) > 0:
        for t, plist in feas_map.items():
            # normalize task key types
            # if keys are not exactly the same as gt_map keys, you may need conversion
            # we try direct, then stringified fallback
            if t not in gt_map and str(t) in gt_map:
                t_key = str(t)
            else:
                t_key = t
            if t_key not in gt_map:
                # Task may be infeasible in this sample; skip if not in GT
                continue
            for p in plist or []:
                feasible_edges.append((t_key, p))
                platform_ids_set.add(p)
    else:
        # Fallback: if feasible is missing, we minimally include the chosen platform as the only neighbor.
        # This degenerates learning (no negatives), but keeps the pipeline functional.
        for t, p in gt_map.items():
            feasible_edges.append((t, p))
            platform_ids_set.add(p)

    # 3) Collect a complete set of platform IDs (from feasible + GT)
    for p in gt_map.values():
        platform_ids_set.add(p)
    platform_ids = list(platform_ids_set)

    return task_ids, platform_ids, feasible_edges, gt_map

In [9]:
# %%
# Feature builders
# If you have real numeric features for tasks/platforms (e.g., ops, mem, device type),
# replace these stubs to return tensors of shape [num_nodes, feat_dim].

def build_task_features(task_ids: List[Any]) -> torch.Tensor:
    # Minimal placeholder: scalar feature = 0.0
    # You can extend: parse task attributes from JSON and stack them here.
    return torch.zeros((len(task_ids), 1), dtype=torch.float32)

def build_platform_features(platform_ids: List[Any]) -> torch.Tensor:
    # Minimal placeholder: scalar feature = 0.0
    # Extend with platform type, memory, etc.
    return torch.zeros((len(platform_ids), 1), dtype=torch.float32)

In [10]:
# %%
# Convert a single result JSON to HeteroData sample

def result_to_graphsample(path: Path) -> Optional[GraphSample]:
    j = read_json(path)
    if j is None:
        return None

    try:
        task_ids, platform_ids, feasible_edges, gt_map = discover_ids_and_edges(j)
    except Exception as e:
        print(f"[WARN] {path.name}: {e}")
        return None

    # Local re-indexing (stable order)
    task_id_map = {tid: i for i, tid in enumerate(task_ids)}
    platform_id_map = {pid: i for i, pid in enumerate(platform_ids)}

    # Node features
    x_task      = build_task_features(task_ids)       # [T, Ft]
    x_platform  = build_platform_features(platform_ids)  # [P, Fp]

    # Edges
    # Build edge index from feasible list (task -> platform)
    edge_t, edge_p = [], []
    for (t_raw, p_raw) in feasible_edges:
        if t_raw in task_id_map and p_raw in platform_id_map:
            edge_t.append(task_id_map[t_raw])
            edge_p.append(platform_id_map[p_raw])

    if len(edge_t) == 0:
        print(f"[WARN] {path.name}: no feasible edges after filtering; skipping.")
        return None

    edge_index = torch.tensor([edge_t, edge_p], dtype=torch.long)  # shape [2, E]

    # Ground-truth: map each task -> platform local index
    gt_local = {}
    missing = 0
    for t_raw, p_raw in gt_map.items():
        if t_raw in task_id_map and p_raw in platform_id_map:
            gt_local[task_id_map[t_raw]] = platform_id_map[p_raw]
        else:
            missing += 1
    if missing > 0:
        print(f"[WARN] {path.name}: {missing} GT mappings could not be localized (skipped).")

    # Build HeteroData
    data = HeteroData()
    data["task"].x = x_task
    data["platform"].x = x_platform

    # task -> platform feasible edges
    data["task", "feasible", "platform"].edge_index = edge_index  # [2, E]

    # Store counts
    data["task"].num_nodes = x_task.size(0)
    data["platform"].num_nodes = x_platform.size(0)

    # Keep GT map as tensors to simplify batching later.
    # We'll store:
    #   - y_task_platform: length T, with -1 for tasks lacking GT
    y = torch.full((x_task.size(0),), fill_value=-1, dtype=torch.long)
    for t_idx, p_idx in gt_local.items():
        y[t_idx] = p_idx
    data["task"].y = y

    # Some metadata
    meta = {
        "result_file": path.name,
        "dataset_id": path.parent.name
    }

    return GraphSample(
        data=data,
        task_id_map=task_id_map,
        platform_id_map=platform_id_map,
        task_gt_platform=gt_local,
        meta=meta
    )

In [30]:
# %%
# FIX: build a list of graph samples — also harden JSON loading to avoid
# "unhashable type: 'list'" by normalizing task/platform IDs into strings.
# -----------------------------------------------------------------------

import json
from typing import Any, Dict, Optional

# 1) Monkey-patch read_json to normalize IDs inside loaded JSONs
_old_read_json = read_json  # from earlier cell

def _norm_id(x: Any) -> str:
    # Ensure all IDs are hashable & comparable (e.g., lists/dicts -> stable JSON string)
    if isinstance(x, (list, dict)):
        try:
            return json.dumps(x, sort_keys=True)
        except Exception:
            return str(x)
    return str(x)

def read_json(path: Path) -> Optional[dict]:
    j = _old_read_json(path)
    if j is None:
        return None

    # Normalize sample.placement_plan keys/values
    sample = j.get("sample", {}) or {}
    if "placement_plan" in sample and isinstance(sample["placement_plan"], dict):
        sample["placement_plan"] = { _norm_id(k): _norm_id(v) for k, v in sample["placement_plan"].items() }
        j["sample"] = sample

    # Normalize feasible maps if present
    for key in ("feasible", "feasible_platforms_by_task", "feasible_map"):
        if key in sample and isinstance(sample[key], dict):
            sample[key] = { _norm_id(k): [ _norm_id(v) for v in (vals or []) ] for k, vals in sample[key].items() }
            j["sample"] = sample

    # Normalize stats.taskResults ids and placement fields if present
    stats = j.get("stats", {}) or {}
    task_results = stats.get("taskResults", []) or []
    for tr in task_results:
        if "id" in tr:
            tr["id"] = _norm_id(tr["id"])
        if "taskId" in tr:
            tr["taskId"] = _norm_id(tr["taskId"])
        # placement could be a string or dict
        placement = tr.get("placement") or tr.get("platform") or tr.get("server")
        if isinstance(placement, dict):
            if "id" in placement:   placement["id"]   = _norm_id(placement["id"])
            if "name" in placement: placement["name"] = _norm_id(placement["name"])
            tr["placement"] = placement
        elif placement is not None:
            tr["placement"] = _norm_id(placement)
    j["stats"] = stats

    return j

# 2) Build graph samples using the existing result_to_graphsample
graph_samples: List[GraphSample] = []
skipped = 0

for p in picked_files:
    try:
        gs = result_to_graphsample(p)
        if gs is not None:
            graph_samples.append(gs)
        else:
            skipped += 1
    except Exception as e:
        skipped += 1
        print(f"[WARN] {p.name}: {e}")

print(f"Prepared {len(graph_samples)} graph samples. Skipped: {skipped}")

# (Optional) quick sanity peek
if graph_samples:
    g0 = graph_samples[0]
    print(
        f"Example sample — tasks={g0.data['task'].num_nodes}, "
        f"platforms={g0.data['platform'].num_nodes}, "
        f"edges={g0.data['task','feasible','platform'].edge_index.size(1)}"
    )

[WARN] simulation_1_placement_8858c9b7409552ee_2e9c24bb.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_8858c9b7409552ee_b8fbe8a1.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_8858c9b7409552ee_f2b42d46.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_c754ee00a31f10a7_f3b19a0d.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_d8151121beae830d_13b89927.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_d8151121beae830d_bc55fd23.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_d8151121beae830d_cdad95e9.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_d8151121beae830d_f9b822b0.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_ff5d622c525e1613_aa1bf76f.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_ff5d622c525e1613_c6a54934.json: maximum recursion depth exceeded
[WARN] simulation_1_placement_ff5d622c525e1613_d0b

In [12]:
# %%
# Train/Val/Test split

def split_list(items, ratios=(0.8, 0.1, 0.1), seed=42):
    assert abs(sum(ratios) - 1.0) < 1e-6
    rnd = random.Random(seed)
    items = items[:]
    rnd.shuffle(items)
    n = len(items)
    n_tr = int(n * ratios[0])
    n_va = int(n * ratios[1])
    return items[:n_tr], items[n_tr:n_tr+n_va], items[n_tr+n_va:]

train_samples, val_samples, test_samples = split_list(graph_samples, (0.8, 0.1, 0.1), seed=42)
len(train_samples), len(val_samples), len(test_samples)

(0, 0, 0)

In [13]:
# %%
# Collation into PyG Data objects
# We will keep each sample as a single HeteroData item (no intra-batch merges), which
# simplifies per-task masked softmax (we iterate over tasks per graph).
#
# If you want bigger batches, PyG can batch hetero graphs, but masked grouping
# must then respect per-graph task id offsets. The code below supports batched
# hetero graphs as well by tracking batch vectors.

def to_pyg_list(samples: List[GraphSample]) -> List[HeteroData]:
    return [s.data for s in samples]

train_data_list = to_pyg_list(train_samples)
val_data_list   = to_pyg_list(val_samples)
test_data_list  = to_pyg_list(test_samples)

BATCH_SIZE = 1  # keep 1 per batch for simplicity and clarity
train_loader = PYGDataLoader(train_data_list, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = PYGDataLoader(val_data_list,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = PYGDataLoader(test_data_list,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Loaders — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [14]:
# %%
# Model
# -----
# GIN backbone per node type with 2-layer MLP encoders (as requested).
# Then score edges via MLP_phi([e_t || e_p]) -> scalar. We then compute
# masked softmax over neighbors N(t) for each task t and apply CE loss.

class TwoLayerMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

def make_gin(in_dim: int, hidden: int, out_dim: int) -> nn.Module:
    # 2-layer MLP as the GIN "nn"
    mlp1 = TwoLayerMLP(in_dim, hidden, hidden)
    mlp2 = TwoLayerMLP(hidden, hidden, out_dim)
    conv1 = GINConv(mlp1, train_eps=True)
    conv2 = GINConv(mlp2, train_eps=True)
    return nn.ModuleList([conv1, conv2])

class AssignmentModel(nn.Module):
    def __init__(
        self,
        task_in: int,
        platform_in: int,
        hid: int = 64,
        out: int = 64,
        dropout: float = 0.0
    ):
        super().__init__()
        # Type encoders (2-layer MLP) for raw node features
        self.task_enc     = TwoLayerMLP(task_in, hid, hid, dropout=dropout)
        self.platform_enc = TwoLayerMLP(platform_in, hid, hid, dropout=dropout)

        # Simple, type-specific GIN stacks (we won’t use edges between same-type nodes,
        # but you can plug homogenous adjacency if you later add them).
        # For now, we'll skip message-passing between same types, and rely on encoders + edge MLP.
        # If you later add intra-type edges, enable GIN here.

        # Edge scoring MLP φ([e_t || e_p]) -> R
        self.edge_mlp = TwoLayerMLP(in_dim=2*hid, hidden=hid, out_dim=1, dropout=dropout)

    def forward(self, data: HeteroData):
        # Encode nodes
        x_t = self.task_enc(data["task"].x)           # [T, hid]
        x_p = self.platform_enc(data["platform"].x)   # [P, hid]

        # Gather pair embeddings for feasible edges
        ei = data["task", "feasible", "platform"].edge_index  # [2, E]
        src_t = ei[0]    # [E]
        dst_p = ei[1]    # [E]

        e_src = x_t[src_t]  # [E, hid]
        e_dst = x_p[dst_p]  # [E, hid]

        scores = self.edge_mlp(torch.cat([e_src, e_dst], dim=-1)).squeeze(-1)  # [E]
        return scores  # one score per feasible edge

In [16]:
# %%
# Masked softmax + loss
# ---------------------
# We need to compute, for EACH TASK, a probability distribution over its neighbor platforms.
# Steps:
# 1) For edge_index (t->p), group edges by t.
# 2) Apply softmax over edges in each group.
# 3) For supervision: convert ground-truth platform index per task into an edge index within
#    that group's edges.

def masked_softmax_per_task(edge_scores: torch.Tensor, edge_index: torch.Tensor, num_tasks: int):
    """
    edge_scores: [E]
    edge_index:  [2, E] with edge_index[0] = task indices (0..T-1)
    Returns:
        probs: [E] softmaxed within each task-group
    """
    t_idx = edge_index[0]  # [E]
    # subtract max per task for numerical stability
    max_per_t = torch.zeros(num_tasks, device=edge_scores.device).index_put_(
        (t_idx,), edge_scores, accumulate=False
    )
    # The above only sets last occurrence. We actually need true max per group.
    max_per_t = torch.full((num_tasks,), -1e30, device=edge_scores.device)
    max_per_t = max_per_t.index_put_((t_idx,), edge_scores, accumulate=True)  # not true max yet

    # Proper groupwise max:
    # We'll do a segmented max using scatter_reduce (PyTorch 2.0+)
    max_per_t = torch.full((num_tasks,), -1e30, device=edge_scores.device)
    max_per_t = torch.scatter_reduce(
        max_per_t, 0, t_idx, edge_scores, reduce="amax", include_self=True
    )

    exps = torch.exp(edge_scores - max_per_t[t_idx])  # [E]
    denom = torch.zeros(num_tasks, device=edge_scores.device)
    denom = torch.scatter_add(denom, 0, t_idx, exps)  # sum per task
    probs = exps / (denom[t_idx] + 1e-12)
    return probs

def compute_loss(
    edge_scores: torch.Tensor,
    data: HeteroData
) -> Tuple[torch.Tensor, float]:
    """
    Cross-entropy over masked softmax per task.
    data["task"].y : [T] with ground-truth local platform index, or -1 if missing.
    """
    ei = data["task", "feasible", "platform"].edge_index  # [2, E]
    t_idx = ei[0]  # [E]
    p_idx = ei[1]  # [E]
    y     = data["task"].y  # [T], values in [0..P-1] or -1

    T = int(data["task"].num_nodes)
    probs = masked_softmax_per_task(edge_scores, ei, T)  # [E]

    # Build per-edge labels: for each task t, the "correct" platform edge is the one
    # whose p_idx equals y[t]. Edges from tasks with y[t] == -1 are ignored.
    valid_mask = (y[t_idx] >= 0)
    correct_edge = (p_idx == y[t_idx]) & valid_mask  # [E] True on the correct edge for each valid task (one-hot per task)

    # To compute CE, we want -log prob(correct_edge) averaged over valid tasks
    # Extract probabilities for correct edges:
    correct_probs = probs[correct_edge]  # [#valid_tasks]
    if correct_probs.numel() == 0:
        # If no supervision in this graph, return zero (skip)
        return torch.tensor(0.0, device=edge_scores.device, requires_grad=True), 0.0

    loss = -torch.log(correct_probs + 1e-12).mean()
    return loss, float(correct_probs.numel())

In [17]:
# %%
# Metrics

@torch.no_grad()
def evaluate(model, loader, device="cpu") -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_count = 0
    total_acc = 0.0

    for data in loader:
        data = data.to(device)
        scores = model(data)  # [E]
        loss, n_valid = compute_loss(scores, data)
        total_loss += float(loss.item()) * max(n_valid, 1)
        total_count += int(n_valid)

        # Top-1 accuracy over tasks with labels
        # For each task, pick argmax over its edges
        ei = data["task", "feasible", "platform"].edge_index
        t_idx, p_idx = ei[0], ei[1]
        T = int(data["task"].num_nodes)
        y = data["task"].y

        # Argmax per task
        # Compute groupwise argmax on scores
        max_scores = torch.full((T,), -1e30, device=scores.device)
        max_scores = torch.scatter_reduce(max_scores, 0, t_idx, scores, reduce="amax", include_self=True)

        # For ties, this picks the last edge with the max; acceptable for evaluation
        argmax_mask = (scores >= max_scores[t_idx] - 1e-12)
        # Among edges flagged by argmax_mask, count correct ones for tasks with labels
        pred_p = p_idx[argmax_mask]        # platforms on predicted edges
        pred_t = t_idx[argmax_mask]        # their tasks
        # reduce to unique tasks (if ties produced multiple)
        # We'll pick the first occurrence per task
        seen = set()
        hits = 0
        denom = 0
        for pt, pp in zip(pred_t.tolist(), pred_p.tolist()):
            if y[pt].item() >= 0 and pt not in seen:
                seen.add(pt)
                denom += 1
                hits += int(pp == y[pt].item())
        if denom > 0:
            total_acc += hits
            # total_count is #valid labels and is already tracked via n_valid

    avg_loss = total_loss / max(total_count, 1)
    avg_acc  = total_acc  / max(total_count, 1)
    return {"loss": avg_loss, "acc": avg_acc, "num_labels": total_count}

In [20]:
# %%
# Instantiate model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Infer feature dims from first sample (fallback to 1 if none)
task_in_dim     = train_data_list[0]["task"].x.size(-1) if len(train_data_list) > 0 else 1
platform_in_dim = train_data_list[0]["platform"].x.size(-1) if len(train_data_list) > 0 else 1

model = AssignmentModel(
    task_in=task_in_dim,
    platform_in=platform_in_dim,
    hid=64,
    out=64,
    dropout=0.0
).to(device)

# Optimizer with defaults (as requested: "don't overthink the parameters")
optimizer = torch.optim.Adam(model.parameters())  # default params

Using device: cpu


In [21]:
# %%
# Training loop

EPOCHS = int(os.environ.get("EPOCHS", "20"))

best_val = {"loss": float("inf"), "acc": 0.0}
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_count = 0

    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        scores = model(data)
        loss, n_valid = compute_loss(scores, data)
        # If a batch has no labels, skip update
        if n_valid == 0:
            continue
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.item()) * n_valid
        epoch_count += int(n_valid)

    avg_train_loss = epoch_loss / max(epoch_count, 1)

    # Eval
    val_stats = evaluate(model, val_loader, device=device)
    print(f"[{epoch:03d}] train_loss={avg_train_loss:.4f} | val_loss={val_stats['loss']:.4f} | val_acc={val_stats['acc']:.4f} (labels={val_stats['num_labels']})")

    # Track best by val_loss
    if val_stats["loss"] < best_val["loss"]:
        best_val = val_stats
        best_state = copy.deepcopy(model.state_dict())

# Load best state if we have one
if best_state is not None:
    model.load_state_dict(best_state)

NameError: name 'train_loader' is not defined

In [22]:
# %%
# Final evaluation on test set

test_stats = evaluate(model, test_loader, device=device)
print(f"[TEST] loss={test_stats['loss']:.4f} | acc={test_stats['acc']:.4f} (labels={test_stats['num_labels']})")

NameError: name 'test_loader' is not defined

In [23]:
# %%
# Inference helper — returns for a single HeteroData graph:
# - per-task top-k candidates (platform indices/local IDs) and probs
# - mapping back to raw platform IDs if you pass the GraphSample

@torch.no_grad()
def predict_topk(
    model: nn.Module,
    data: HeteroData,
    k: int = 3,
    device: str = "cpu"
) -> Dict[int, List[Tuple[int, float]]]:
    model.eval()
    data = data.to(device)
    scores = model(data)  # [E]
    ei = data["task", "feasible", "platform"].edge_index
    t_idx, p_idx = ei[0], ei[1]
    T = int(data["task"].num_nodes)

    probs = masked_softmax_per_task(scores, ei, T)  # [E]

    # collect edges per task
    out: Dict[int, List[Tuple[int, float]]] = {t: [] for t in range(T)}
    for e, (t, p) in enumerate(zip(t_idx.tolist(), p_idx.tolist())):
        out[t].append((p, float(probs[e].item())))

    # sort and trim top-k per task
    for t in range(T):
        out[t].sort(key=lambda x: x[1], reverse=True)
        out[t] = out[t][:k]
    return out

In [ ]:
# %%
# Example predictions on a few test graphs

if len(test_samples) > 0:
    k = 3
    for i, gs in enumerate(test_samples[:5], start=1):
        pred = predict_topk(model, gs.data, k=k, device=device)
        print(f"\n--- Test sample {i} — {gs.meta.get('result_file')} ---")
        # map local platform indices back to raw IDs
        inv_platform = {v: k for k, v in gs.platform_id_map.items()}

        for t_local in range(gs.data["task"].num_nodes):
            picks = [(inv_platform[p], prob) for (p, prob) in pred[t_local]]
            y = gs.task_gt_platform.get(t_local, None)
            y_raw = inv_platform[y] if y is not None else None
            print(f"Task[{t_local}]  GT={y_raw}  top{k}={picks}")

In [25]:
# %%
# Save artifacts

ARTIFACTS_DIR = Path("./artifacts_gnn_t2p")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACTS_DIR / "assignment_gin.pt"
torch.save(model.state_dict(), model_path)
print(f"Saved model to {model_path}")

# Save a minimal config/metadata for reproducibility
meta = {
    "base_dir": str(BASE_DIR),
    "datasets_dir": str(DATASETS_DIR),
    "results_dir": str(RESULTS_DIR),
    "epochs": EPOCHS,
    "model": "GIN + 2-layer encoders, edge-MLP",
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open(ARTIFACTS_DIR / "run_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"Saved run meta to {ARTIFACTS_DIR / 'run_meta.json'}")

Saved model to artifacts_gnn_t2p/assignment_gin.pt
Saved run meta to artifacts_gnn_t2p/run_meta.json


In [26]:
# %%
# (Optional) Export a small CSV of test predictions vs labels for quick inspection

def dump_test_predictions(samples: List[GraphSample], path_csv: Path, k=3):
    rows = []
    for gs in samples:
        pred = predict_topk(model, gs.data, k=k, device=device)
        inv_platform = {v: k for k, v in gs.platform_id_map.items()}
        for t_local in range(gs.data["task"].num_nodes):
            topk = pred[t_local]
            y = gs.task_gt_platform.get(t_local, None)
            y_raw = inv_platform[y] if y is not None else None
            row = {
                "result_file": gs.meta.get("result_file"),
                "task_local": t_local,
                "gt_platform": y_raw
            }
            for j, (p_idx, prob) in enumerate(topk, start=1):
                row[f"top{j}_platform"] = inv_platform[p_idx]
                row[f"top{j}_prob"] = prob
            rows.append(row)
    pd.DataFrame(rows).to_csv(path_csv, index=False)

pred_csv = ARTIFACTS_DIR / "test_topk.csv"
dump_test_predictions(test_samples, pred_csv, k=3)
print(f"Wrote {pred_csv}")

Wrote artifacts_gnn_t2p/test_topk.csv


In [27]:
# %%
# Extensions & hooks (quick notes)
# --------------------------------
# 1) Real node features:
#    - In discover_ids_and_edges, also parse per-task type, shape, size, client ID, etc.
#    - In build_task_features / build_platform_features, assemble numeric vectors accordingly.
# 2) Edge features:
#    - If you later add edge features (latency, bandwidth), change edge_mlp to take
#      [e_t || e_p || e_tp] and wire them into the forward.
# 3) Larger batches:
#    - You can batch multiple graphs and adapt masked-softmax to use batch vectors:
#      data["task"].batch gives graph IDs per task in the batch.
# 4) Loss reweighting:
#    - If tasks have class imbalance (some platforms underrepresented), you can add
#      per-task weights in the CE aggregation.
# 5) Export top-k placement plans:
#    - Use predict_topk to build candidate placements for new workloads; feed them into
#      your simulation driver to test end-to-end RTT.